# Occam's Reader Holdings Upload Merge Script

### About

Alma Analytics is used to generate bibliographic metadata for all active portfolios in approved ebook collections for upload to Occam's Reader. The Analytics report provides title, author, edition, publication date, ISBN, OCLC number, publisher, and portfolio ID.

Because Alma Analytics does not provide the resolved target URL needed for Occam's Reader, an Export URLs job is run against the same population of portfolios. The Export URLs output includes the portfolio ID and the corresponding title's URL.

The metadata export and URL export are merged by this script using the Portfolio ID as the matching key. The resulting file is formatted according to the Occam's Reader holdings template and exported as a CSV for upload.

In [58]:
# Import required libraries

import polars as pl
from polars import col

from datetime import date

In [59]:
# Variables

# Change these based on input file names
metadata_csv = "input/metadata analytics.csv"
urls_csv = "input/urls.csv"

# Leave as is
today = date.today()

### Ingest Data

In [60]:
# Ingest metadata
df_metadata = pl.read_csv(
    metadata_csv,
    schema_overrides={
        "OCLC Control Number (035a)": pl.String
    }
)

# Ingest urls
df_urls = pl.read_csv(urls_csv)

### Transform Data

In [61]:
# Transform metadata
df_metadata = df_metadata.rename({"Portfolio Id":"ID"})


# Transform urls
df_urls = df_urls.with_columns(
    col("ID").str.strip_chars("'").cast(pl.Int64)
)

### Merge data

In [62]:
df_merged = df_metadata.join(df_urls, on="ID", how="left")

### Restructure Merged Data

In [63]:
df_final = df_merged.select(
    col("Title"),
    col("Author"),
    col("Edition"),
    col("Publication Date").alias("Year"),
    col("ISBN"),
    col("OCLC Control Number (035a)").alias("OCLC Number"),
    col("Publisher"),
    col("URL")
)

### Export Data to CSV

In [64]:
df_final.write_csv(f"output/occamHoldings_{today}.csv")